# Exact SR-GNN / TAGNN Dataset Preprocessing

This notebook implements the preprocessing pipeline used for **Yoochoose** and **Diginetica** in SR-GNN and TAGNN.

Implemented steps:
1. Reconstruct ordered sessions from compressed raw chunks.
2. Drop sessions with length `< 2`.
3. Count global item frequency on the remaining sessions.
4. Remove items with frequency `< 5`.
5. Drop sessions that become length `< 2` after item filtering.
6. Sort sessions chronologically.
7. Split by date using the paper-compatible rule:
   - **Yoochoose**: last **1 day** is test
   - **Diginetica**: last **7 days** is test
8. Remove from the **test** split all items unseen in training.
9. Drop test sessions that become length `< 2`.
10. For **Yoochoose**, optionally keep only the most recent **1/64** of the training sessions.
11. Save one compressed artifact per dataset with final train/test sessions.
12. Provide helpers to generate prefix-label examples lazily:
   `([v1], v2), ([v1, v2], v3), ...`

Notes:
- The split rule intentionally matches the official preprocessing implementation:
  `train_date < split_date` and `test_date > split_date`.
  Sessions exactly on `split_date` are excluded.
- Validation splitting is **not** part of raw preprocessing, but an optional helper is included.

In [ ]:
import gzip
import pickle
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Iterator

import pandas as pd

Session = tuple[int, pd.Timestamp, list[int]]

PROJECT_ROOT = Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

YOOCHOOSE_RAW = RAW_DIR / "yoochoose" / "yoochoose-clicks.pkl.gz"
DIGINETICA_RAW = RAW_DIR / "diginetica" / "train-item-views.pkl.gz"

print(f"Yoochoose compressed raw file exists: {YOOCHOOSE_RAW.exists()}")
print(f"Diginetica compressed raw file exists: {DIGINETICA_RAW.exists()}")


@dataclass(frozen=True)
class DatasetConfig:
    name: str
    raw_path: Path
    test_days: int
    keep_train_fraction_denominator: int | None = None
    min_item_freq: int = 5

In [ ]:
def iter_chunked_pickle(path: Path) -> Iterator[pd.DataFrame]:
    """Yield DataFrame chunks from a gzipped pickle stream."""
    with gzip.open(path, "rb") as f:
        metadata = pickle.load(f)
        if not isinstance(metadata, dict):
            raise ValueError(f"Invalid metadata block in {path}")

        while True:
            try:
                chunk = pickle.load(f)
            except EOFError:
                break

            if not isinstance(chunk, pd.DataFrame):
                raise ValueError(
                    f"Expected DataFrame chunk in {path}, got {type(chunk)!r}"
                )

            yield chunk


def load_yoochoose_sessions(path: Path) -> list[Session]:
    """Build ordered Yoochoose sessions from chunked raw clicks."""
    events_by_session: dict[int, list[tuple[pd.Timestamp, int]]] = defaultdict(list)

    for chunk in iter_chunked_pickle(path):
        chunk = chunk[["session_id", "timestamp", "item_id"]].copy()
        chunk["timestamp"] = pd.to_datetime(chunk["timestamp"], utc=True).dt.tz_convert(
            None
        )
        chunk["item_id"] = chunk["item_id"].astype("int64")

        for sid, part in chunk.groupby("session_id", sort=False):
            events_by_session[int(sid)].extend(
                zip(part["timestamp"].tolist(), part["item_id"].tolist())
            )

    sessions: list[Session] = []
    for sid, events in events_by_session.items():
        events.sort(key=lambda x: x[0])
        session_date = events[-1][0].normalize()
        items = [item for _, item in events]
        sessions.append((sid, session_date, items))

    return sessions


def load_diginetica_sessions(path: Path) -> list[Session]:
    """Build ordered Diginetica sessions from chunked raw item views."""
    events_by_session: dict[int, list[tuple[int, int]]] = defaultdict(list)
    session_dates: dict[int, pd.Timestamp] = {}

    for chunk in iter_chunked_pickle(path):
        chunk = chunk[["session_id", "eventdate", "timeframe", "item_id"]].copy()
        chunk["eventdate"] = pd.to_datetime(chunk["eventdate"], format="%Y-%m-%d")
        chunk["timeframe"] = chunk["timeframe"].astype("int64")
        chunk["item_id"] = chunk["item_id"].astype("int64")

        for sid, part in chunk.groupby("session_id", sort=False):
            sid = int(sid)
            events_by_session[sid].extend(
                zip(part["timeframe"].tolist(), part["item_id"].tolist())
            )

            # A session should belong to its latest day.
            latest_date = pd.Timestamp(part["eventdate"].max()).normalize()
            previous_date = session_dates.get(sid)
            session_dates[sid] = (
                latest_date
                if previous_date is None
                else max(previous_date, latest_date)
            )

    sessions: list[Session] = []
    for sid, events in events_by_session.items():
        events.sort(key=lambda x: x[0])
        items = [item for _, item in events]
        sessions.append((sid, session_dates[sid], items))

    return sessions


def session_stats(sessions: list[Session]) -> dict[str, float]:
    clicks = sum(len(items) for _, _, items in sessions)
    session_count = len(sessions)
    unique_items = len({item for _, _, items in sessions for item in items})
    avg_len = (clicks / session_count) if session_count else 0.0
    return {
        "clicks": clicks,
        "sessions": session_count,
        "items": unique_items,
        "avg_len": avg_len,
    }


def print_stats(title: str, sessions: list[Session]) -> None:
    stats = session_stats(sessions)
    print(
        f"{title:24s} | clicks={stats['clicks']:,} | sessions={stats['sessions']:,} | "
        f"items={stats['items']:,} | avg_len={stats['avg_len']:.2f}"
    )


def filter_sessions_step6(
    sessions: list[Session], min_item_freq: int = 5
) -> list[Session]:
    """Exact shared filtering used before train/test splitting."""
    sessions = [(sid, date, items) for sid, date, items in sessions if len(items) >= 2]

    item_frequency = Counter()
    for _, _, items in sessions:
        item_frequency.update(items)

    filtered: list[Session] = []
    for sid, date, items in sessions:
        kept_items = [item for item in items if item_frequency[item] >= min_item_freq]
        if len(kept_items) >= 2:
            filtered.append((sid, date, kept_items))

    return filtered


def sort_sessions_by_date(sessions: list[Session]) -> list[Session]:
    return sorted(sessions, key=lambda x: (x[1], x[0]))


def split_sessions_by_date(
    sessions: list[Session], test_days: int
) -> tuple[list[Session], list[Session], pd.Timestamp]:
    """Match the official split rule: train < split_date, test > split_date."""
    if not sessions:
        raise ValueError("Cannot split an empty session list.")

    max_date = max(date for _, date, _ in sessions)
    split_date = max_date - pd.Timedelta(days=test_days)

    train_sessions = [session for session in sessions if session[1] < split_date]
    test_sessions = [session for session in sessions if session[1] > split_date]

    return train_sessions, test_sessions, split_date


def filter_test_to_seen_train_items(
    train_sessions: list[Session],
    test_sessions: list[Session],
) -> list[Session]:
    """Remove test items not seen in training, then drop short sessions again."""
    train_items = {item for _, _, items in train_sessions for item in items}

    cleaned_test: list[Session] = []
    for sid, date, items in test_sessions:
        kept_items = [item for item in items if item in train_items]
        if len(kept_items) >= 2:
            cleaned_test.append((sid, date, kept_items))

    return cleaned_test


def keep_most_recent_fraction(
    sessions: list[Session],
    denominator: int | None,
) -> list[Session]:
    """Keep the most recent 1/denominator part of sessions."""
    if denominator is None:
        return sessions

    if denominator <= 0:
        raise ValueError("denominator must be positive")

    keep_count = len(sessions) // denominator
    if keep_count == 0:
        return sessions

    return sessions[-keep_count:]


def count_prefix_examples(sessions: list[Session]) -> int:
    return sum(len(items) - 1 for _, _, items in sessions)


def iter_prefix_examples(sessions: list[Session]):
    """Yield (input_sequence, label) pairs lazily."""
    for _, _, items in sessions:
        for end_idx in range(1, len(items)):
            yield items[:end_idx], items[end_idx]


def make_validation_split(
    train_sessions: list[Session],
    valid_fraction: float = 0.1,
    random_state: int = 42,
) -> tuple[list[Session], list[Session]]:
    """Optional helper for hyperparameter tuning; not part of raw preprocessing."""
    if not 0.0 < valid_fraction < 1.0:
        raise ValueError("valid_fraction must be between 0 and 1")

    shuffled = (
        pd.Series(train_sessions).sample(frac=1.0, random_state=random_state).tolist()
    )
    valid_size = int(len(shuffled) * valid_fraction)
    valid_sessions = shuffled[:valid_size]
    new_train_sessions = shuffled[valid_size:]
    return new_train_sessions, valid_sessions


def serialize_sessions(sessions: list[Session]) -> dict[str, list]:
    return {
        "session_ids": [sid for sid, _, _ in sessions],
        "session_dates": [date.strftime("%Y-%m-%d") for _, date, _ in sessions],
        "session_item_sequences": [items for _, _, items in sessions],
    }


def save_preprocessed_dataset(
    output_path: Path,
    config: DatasetConfig,
    step6_sessions: list[Session],
    train_sessions: list[Session],
    test_sessions: list[Session],
) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    payload = {
        "dataset": config.name,
        "config": {
            "name": config.name,
            "raw_path": str(config.raw_path),
            "test_days": config.test_days,
            "keep_train_fraction_denominator": config.keep_train_fraction_denominator,
            "min_item_freq": config.min_item_freq,
        },
        "step6_stats": session_stats(step6_sessions),
        "train_stats": session_stats(train_sessions),
        "test_stats": session_stats(test_sessions),
        "train_example_count": count_prefix_examples(train_sessions),
        "test_example_count": count_prefix_examples(test_sessions),
        "train": serialize_sessions(train_sessions),
        "test": serialize_sessions(test_sessions),
    }

    with gzip.open(output_path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
def preprocess_dataset(
    config: DatasetConfig,
    session_loader: Callable[[Path], list[Session]],
) -> dict:
    print(f"\n===== {config.name.upper()} =====")
    print(f"Reading compressed file: {config.raw_path}")

    if not config.raw_path.exists():
        raise FileNotFoundError(
            f"Missing input file: {config.raw_path}. Run the download notebook first."
        )

    sessions = session_loader(config.raw_path)
    print_stats("raw sessions", sessions)

    step6_sessions = filter_sessions_step6(sessions, min_item_freq=config.min_item_freq)
    step6_sessions = sort_sessions_by_date(step6_sessions)
    print_stats("after step 6 filtering", step6_sessions)

    train_sessions, test_sessions, split_date = split_sessions_by_date(
        step6_sessions,
        test_days=config.test_days,
    )
    print(f"split_date = {split_date.strftime('%Y-%m-%d')}")

    print_stats("train before test cleanup", train_sessions)
    print_stats("test before test cleanup", test_sessions)

    test_sessions = filter_test_to_seen_train_items(train_sessions, test_sessions)
    print_stats("test after test cleanup", test_sessions)

    train_sessions = keep_most_recent_fraction(
        train_sessions,
        denominator=config.keep_train_fraction_denominator,
    )
    if config.keep_train_fraction_denominator is not None:
        print_stats(
            f"train after 1/{config.keep_train_fraction_denominator} keep",
            train_sessions,
        )

    result = {
        "config": config,
        "split_date": split_date,
        "step6_sessions": step6_sessions,
        "train_sessions": train_sessions,
        "test_sessions": test_sessions,
        "step6_stats": session_stats(step6_sessions),
        "train_stats": session_stats(train_sessions),
        "test_stats": session_stats(test_sessions),
        "train_example_count": count_prefix_examples(train_sessions),
        "test_example_count": count_prefix_examples(test_sessions),
    }

    return result

In [ ]:
yoochoose_config = DatasetConfig(
    name="yoochoose_1_64",
    raw_path=YOOCHOOSE_RAW,
    test_days=1,
    keep_train_fraction_denominator=64,
)

diginetica_config = DatasetConfig(
    name="diginetica",
    raw_path=DIGINETICA_RAW,
    test_days=7,
    keep_train_fraction_denominator=None,
)

yoochoose_result = preprocess_dataset(yoochoose_config, load_yoochoose_sessions)
diginetica_result = preprocess_dataset(diginetica_config, load_diginetica_sessions)

yoochoose_out = PROCESSED_DIR / "yoochoose_1_64_preprocessed.pkl.gz"
diginetica_out = PROCESSED_DIR / "diginetica_preprocessed.pkl.gz"

save_preprocessed_dataset(
    output_path=yoochoose_out,
    config=yoochoose_config,
    step6_sessions=yoochoose_result["step6_sessions"],
    train_sessions=yoochoose_result["train_sessions"],
    test_sessions=yoochoose_result["test_sessions"],
)

save_preprocessed_dataset(
    output_path=diginetica_out,
    config=diginetica_config,
    step6_sessions=diginetica_result["step6_sessions"],
    train_sessions=diginetica_result["train_sessions"],
    test_sessions=diginetica_result["test_sessions"],
)

print("\nSaved files:")
print(f" - {yoochoose_out}")
print(f" - {diginetica_out}")

In [ ]:
summary = pd.DataFrame(
    [
        {
            "dataset": "yoochoose_1_64",
            "step6_sessions": yoochoose_result["step6_stats"]["sessions"],
            "step6_items": yoochoose_result["step6_stats"]["items"],
            "train_sessions": yoochoose_result["train_stats"]["sessions"],
            "test_sessions": yoochoose_result["test_stats"]["sessions"],
            "train_examples": yoochoose_result["train_example_count"],
            "test_examples": yoochoose_result["test_example_count"],
        },
        {
            "dataset": "diginetica",
            "step6_sessions": diginetica_result["step6_stats"]["sessions"],
            "step6_items": diginetica_result["step6_stats"]["items"],
            "train_sessions": diginetica_result["train_stats"]["sessions"],
            "test_sessions": diginetica_result["test_stats"]["sessions"],
            "train_examples": diginetica_result["train_example_count"],
            "test_examples": diginetica_result["test_example_count"],
        },
    ]
)

summary

In [ ]:
with gzip.open(yoochoose_out, "rb") as f:
    yoochoose_payload = pickle.load(f)

with gzip.open(diginetica_out, "rb") as f:
    diginetica_payload = pickle.load(f)

print("Yoochoose payload keys:", sorted(yoochoose_payload.keys()))
print("Diginetica payload keys:", sorted(diginetica_payload.keys()))
print(
    "Example Yoochoose train session:",
    yoochoose_payload["train"]["session_item_sequences"][0][:10],
)

first_three_examples = []
for idx, example in enumerate(iter_prefix_examples(yoochoose_result["train_sessions"])):
    first_three_examples.append(example)
    if idx == 2:
        break

print("First 3 lazy prefix examples:", first_three_examples)